<a href="https://colab.research.google.com/github/EstherMan05/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [8]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [9]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print(total_revenue, total_units)

8520.0 783


Interpretation: across 400 orders this game, vendors sold 783 individual items for a combined $8,520 in revenue which is an average of about $21.30 per order.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [10]:
by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).reset_index()
by_category['share_pct'] = (by_category['revenue'] / by_category['revenue'].sum() * 100).round(1)
print(by_category)

   category  revenue  share_pct
0      Food   4293.0       50.4
1     Merch   1771.5       20.8
2     Drink   1554.0       18.2
3  RainGear    901.5       10.6


Interpretation: Food alone accounts for just over half of all revenue ($4,293, 50.4%), while RainGear is the smallest category at 10.6%, which is unsurprising given it was also the rarest category by sampling probability (10%).

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [11]:
by_vendor = df.groupby('vendor_id')['revenue'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(by_vendor)

                mean  count
vendor_id                  
V-01       22.595745     94
V-18       21.750000    108
V-05       20.580645     93
V-10       20.314286    105


Interpretation: V-01 has the highest average order revenue ($22.60), but it's backed by a solid 94 orders, so this isn't a fluke from a tiny sample, all four vendors have comparable, substantial order counts (93–108), so the ranking is trustworthy.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [12]:
merch_share = df.loc[df['category'] == 'Merch', 'revenue'].sum() / df['revenue'].sum() * 100
print(round(merch_share, 1))

20.8


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [13]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

assert len(joined) == len(df)
assert abs(joined['revenue'].sum() - df['revenue'].sum()) < 0.01

unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
print(unmatched)

joined['vendor_name'] = joined['vendor_name'].fillna(joined['vendor_id'])

['V-18']


**The unmatched vendor, and what I did about it:** _..._

V-18 has no matching name in the lookup table. Since it's a real, substantial vendor (108 orders — the highest count of any vendor), dropping it would meaningfully understate the report. I filled its missing vendor_name with its vendor_id ("V-18") as a placeholder so it stays visible in the report rather than silently disappearing or showing as NaN.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [14]:
pivot = pd.pivot_table(
    joined, index='vendor_name', columns='category', values='revenue',
    aggfunc='sum', fill_value=0, margins=True, margins_name='Total'
)
print(pivot)

category          Drink    Food   Merch  RainGear   Total
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
V-18              582.0  1018.5   508.5     240.0  2349.0
Total            1554.0  4293.0  1771.5     901.5  8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [15]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a)
The vendors should focus on maintaining strong Food sales while looking for opportunities to increase revenue from the lower-performing categories. Food generated $4,293, or 50.4% of total revenue, so it was clearly an important source of revenue. At the same time, RainGear generated only $901.50, or 10.6%, suggesting vendors could consider offering more RainGear products or promoting them more heavily next game. Merch also contributed 20.8% of revenue, so vendors should continue selling it while exploring ways to increase sales across categories.

b)
The least trustworthy answer is Q3, which compares vendors based on average order revenue. Although the vendors have reasonably similar numbers of orders, an average can still be affected by the particular mix of prices and quantities in each vendor's orders. For example, the dataset randomly assigns prices and quantities, so a vendor could have a higher average simply because it happened to receive more expensive or larger orders. Therefore, the ranking of vendors by average order revenue should be treated cautiously rather than as evidence that one vendor consistently performs better.